In [ ]:
# K-means clustering — geological workflow (without FI)
# Groups all intervals into petrophysical populations using TOC, PI, and Tmax.
# Reproducible: z-score standardization, k-means++, n_init = 50, random_state = 42.

import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# Load data (calculated outputs for all wells)
file_path = "Outputs_of_all_wells.xlsx"
df = pd.read_excel(file_path, sheet_name="Per well normalized")

features = ["TOC", "PI", "Tmax"]
for col in ["Well Number", "API", "Depth"] + features:
    df[col] = pd.to_numeric(df[col], errors="coerce")
df_clean = df.dropna(subset=["Well Number", "API", "Depth"] + features).copy()

# Standardize and cluster (K = 4)
X = df_clean[features].values
X_scaled = StandardScaler().fit_transform(X)

k = 4
kmeans = KMeans(n_clusters=k, random_state=42, n_init=50)
df_clean["KM_Cluster_without_FI"] = kmeans.fit_predict(X_scaled)

sil = silhouette_score(X_scaled, df_clean["KM_Cluster_without_FI"])
print("Rows used:", len(df_clean), "| Wells:", df_clean["Well Number"].nunique())
print("Silhouette score (K=4):", round(sil, 4))

# Cluster centroids (in original units)
centers = pd.DataFrame(
    StandardScaler().fit(X).inverse_transform(kmeans.cluster_centers_),
    columns=["Mean_TOC", "Mean_PI", "Mean_Tmax"]
)
centers["Count"] = df_clean["KM_Cluster_without_FI"].value_counts().sort_index().values
print("\nCluster centroids:")
print(centers.round(4))

# Assign population names.
# Mapping is verified from the cluster centroids:
#   Cluster 0 -> low TOC (1.53)                    -> Low-Quality
#   Cluster 1 -> highest TOC (2.22), moderate PI   -> Organic-Rich
#   Cluster 2 -> high PI (0.35), low TOC           -> High-PI
#   Cluster 3 -> high TOC (2.08), low PI (0.08)   -> Organic-Rich Low-PI
population_names = {
    0: "Low-Quality",
    1: "Organic-Rich",
    2: "High-PI",
    3: "Organic-Rich Low-PI"
}
df_clean["Population_Name"] = df_clean["KM_Cluster_without_FI"].map(population_names)

# Population proportions
print("\nPopulation proportions (%):")
print((df_clean["Population_Name"].value_counts(normalize=True) * 100).round(1))

# The interpreted clustering results (cluster assignments and population names)
# are provided as a separate file in the outputs/ folder of this repository.

# K-selection diagnostics: silhouette and inertia for k = 2 to 10 (justifies K = 4)
print("\nK-selection diagnostics:")
for kk in range(2, 11):
    km = KMeans(n_clusters=kk, random_state=42, n_init=50).fit(X_scaled)
    print(f"  k = {kk}: silhouette = {silhouette_score(X_scaled, km.labels_):.4f} | "
          f"inertia = {km.inertia_:.1f}")